In [1]:
import torch
import torchvision
torchvision.disable_beta_transforms_warning()
from torchvision.transforms import v2
from torchvision.models import efficientnet_b0,EfficientNet_B0_Weights,densenet121,DenseNet121_Weights
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import random
import os
import timm
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score, ConfusionMatrixDisplay
from copy import deepcopy
from typing import Tuple, List, Dict
import csv
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class KONet(torch.nn.Module):

    def __init__(
            self,
            m1_ratio=0.4,
            m2_ratio=0.6,
            m1_dropout=0.1,
            m2_dropout=0.3,
            n_classes=2
    ):
        super().__init__()
        assert m1_ratio+m2_ratio==1
        self.n_classes=n_classes
        self.m1_ratio=m1_ratio #EfficientNet
        self.m2_ratio=m2_ratio #DenseNet
        self.m1_dropout=m1_dropout
        self.m2_dropout=m2_dropout

        self.efficient=efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.efficient.classifier[0]=torch.nn.Dropout(p=self.m1_dropout,inplace=True)
        self.efficient.classifier[-1]=torch.nn.Linear(in_features=1280,out_features=self.n_classes)

        self.dense=densenet121(weights=DenseNet121_Weights.DEFAULT)
        self.dense.classifier=torch.nn.Sequential(torch.nn.Dropout(p=self.m2_dropout,inplace=True),
                                            torch.nn.Linear(in_features=1024,out_features=n_classes),
                                            )

    def forward(self, x):
        m1=self.efficient(x)
        m2=self.dense(x)
        out=self.m1_ratio*m1+self.m2_ratio*m2
        return out


def test(model,dataloader,loss_fn):
    model.eval()
    loss=0
    labels=[]
    probabilities=[]
    for data,label in dataloader:
        with torch.no_grad():
            data , label=data.to(device) , label.to(device)

            output=model(data)
            loss+=loss_fn(output , label)
            prob=output.softmax(dim=1)
            labels.append(label.detach().cpu().numpy())
            probabilities.append(prob.detach().cpu().numpy())

    labels=np.concatenate(labels,axis=0)
    probabilities=np.concatenate(probabilities,axis=0)

    loss=loss/len(dataloader)
    return loss.item(),labels,probabilities


def set_random_seed(seed: int = 2222, deterministic: bool = False):
        random.seed(seed)
        np.random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)  # type: ignore
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = deterministic  # type: ignore

class CustomImageFolder(torchvision.datasets.ImageFolder):
    def __init__(
        self,
        root,
        transform= None,
        class_map = None
    ):
        self.map = class_map
        super().__init__(
            root,
            transform=transform,
        )
        

    def find_classes(self, directory: str) -> Tuple[List[str], Dict[str, int]]:
        """
        Override this method to load from setting file instead of scanning directory
        """
        classes = list(self.map.keys())
        classes_to_idx = self.map
        return classes, classes_to_idx
def prep_dataset(path, image_shape=224, augmented_dataset_size=1800,class_map=None,
                 train_split=0.8, valid_split=0.1, test_split=0.1, seed=42):

    non_augment_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32,scale=True),
        v2.Resize((image_shape, image_shape), antialias=True),
        v2.Normalize(mean=[0.5], std=[0.5]),
    ])

    aug_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32,scale=True),
        v2.RandomAffine(degrees=30, shear=30),
        v2.RandomZoomOut(side_range=(1, 1.5)),
        v2.Resize((image_shape, image_shape), antialias=True),
        v2.Normalize(mean=[0.5], std=[0.5]),
    ])
    base_dataset = CustomImageFolder(path, transform=non_augment_transform,class_map=class_map)
    augmented_dataset = CustomImageFolder(path, transform=aug_transform,class_map=class_map)
    n_samples = len(base_dataset)

    labels = [s[1] for s in base_dataset.samples]
    indices = np.arange(n_samples)

    # split test first
    train_val_idx, test_idx = train_test_split(indices, test_size=test_split, stratify=labels, random_state=seed)

    kf = KFold(n_splits=10, shuffle=True, random_state=seed)
    splits = list(kf.split(train_val_idx))

    fold_datasets=[]
    factor=augmented_dataset_size//len(base_dataset)
    
    for fold, (train_indices, val_indices) in enumerate(splits):
        augmented_train_indices = np.repeat(train_indices, factor)
        train_set = torch.utils.data.Subset(augmented_dataset, augmented_train_indices)
        valid_set = torch.utils.data.Subset(base_dataset, val_indices)
        fold_datasets.append((fold+1,train_set,valid_set))

    test_subset = torch.utils.data.Subset(base_dataset, test_idx)


    return fold_datasets, test_subset

In [10]:
n_classes=3
image_shape=224
augmented_dataset_size=1800
batch_size=32
epochs=20
ewc_lambda=16
seed=42
model_name= 'mobilenet'
script_name = 'train osteopenia incremental early stop'
results_file = "fold_results.csv"
path1="/kaggle/input/datasets/haiderazam/kod-2-class/Osteoporosis Knee X-ray Preprocessed"
path2="/kaggle/input/datasets/haiderazam/kod-only-osteopenia/Osteoporosis Knee X-ray only osteopenia Preprocessed"
weigths_path="/kaggle/input/datasets/haiderazam/osteopenia-incremental-early-stop-weights/train osteopenia incremental early stop/model"
os.makedirs('model',exist_ok=True)

In [11]:
map1={'normal':0,'osteoporosis':2}
map2={'osteopenia':1}

set_random_seed(seed)

fold_datasets1,test_set1=prep_dataset(path1,image_shape,augmented_dataset_size,map1)
fold_datasets2,test_set2=prep_dataset(path2,image_shape,augmented_dataset_size,map2)

best_acc=0
best_loss=np.inf
print('Model: ',model_name)
for (fold,train_set1,valid_set1),(_,train_set2,valid_set2) in zip(fold_datasets1,fold_datasets2):
    print(f"Fold {fold}")

    print(f"Test set 1 size: {len(test_set1)}, Test set 2 size: {len(test_set2)}")

    test_set = torch.utils.data.ConcatDataset([test_set1, test_set2])
    
    test_dataloader = DataLoader(test_set, batch_size=batch_size, num_workers=4, pin_memory=True,
                                    persistent_workers=True, shuffle=False)
    
    if 'efficient' in model_name:
        model=efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        p=0.1
        model.classifier[0]=torch.nn.Dropout(p=p,inplace=True)
        model.classifier[-1]=torch.nn.Linear(in_features=1280,out_features=n_classes)
    
    elif 'dense' in model_name:
        model=densenet121(weights=DenseNet121_Weights.DEFAULT)
        p=0.3
        model.classifier=torch.nn.Sequential(torch.nn.Dropout(p=p,inplace=True),
                                            torch.nn.Linear(in_features=1024,out_features=n_classes),
                                            )
    
    elif 'conv_next' in model_name:
        p=0.3
        model=torchvision.models.convnext_tiny(weights='DEFAULT')
        model.classifier[2]=torch.nn.Sequential(torch.nn.Dropout(p=p,inplace=True),
                                            torch.nn.Linear(in_features=768,out_features=n_classes),
                                            )
    elif 'edgenext' in model_name:
        model=timm.create_model("edgenext_small.usi_in1k",
                                        pretrained=True, 
                                        features_only=False,
                                        in_chans=3,
                                        num_classes=n_classes,
                                        global_pool='avg'
                                        )
    
    elif 'KONet' in model_name:
        m1_ratio=0.6
        m2_ratio=0.4
        m1_dropout=0.1
        m2_dropout=0.3
        model=KONet(m1_ratio=m1_ratio,m2_ratio=m2_ratio,m1_dropout=m1_dropout,m2_dropout=m2_dropout,n_classes=n_classes)
    
    elif 'mobilenet' in model_name:
        model=torchvision.models.mobilenet_v3_small(weights='DEFAULT')
        model.classifier[3]=torch.nn.Linear(in_features=1024,out_features=n_classes)

    
    model.load_state_dict(torch.load(f'{weigths_path}/{model_name}_fold_{fold}_best_param.pkl'))

    
    model.to(device)
    #optimizer=torch.optim.AdamW(model.parameters(),lr=0.0000625)
    optimizer=torch.optim.SGD(model.parameters(),lr=0.000008,momentum=0.9, weight_decay=5e-4)
    
    loss_fn=torch.nn.CrossEntropyLoss()

    model.eval()
    test_loss1, test_labels, test_probabilities = test(model, test_dataloader, loss_fn)
    test_pred_labels = np.argmax(test_probabilities, axis=1)
    test_acc1 = balanced_accuracy_score(test_labels, test_pred_labels)
    test_auc=roc_auc_score(test_labels,test_probabilities,average='weighted',multi_class='ovr')
    test_f1=f1_score(test_labels,test_pred_labels,average='weighted')
    if best_acc<test_acc1:
        best_acc=test_acc1
        print(f"\nBest accuracy at fold {fold}")
    if best_loss>test_loss1:
        best_loss=test_loss1
        print(f"\nBest loss at fold {fold}")
    print(f"Combined Test set accuracy: {test_acc1}")
    print(f"F1-Score: {test_f1}")
    print(f"ROC_AUC: {test_auc}")


Model:  mobilenet
Fold 1
Test set 1 size: 14, Test set 2 size: 20
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 150MB/s]



Best accuracy at fold 1

Best loss at fold 1
Combined Test set accuracy: 0.3
F1-Score: 0.4072398190045249
ROC_AUC: 0.44691338073691017
Fold 2
Test set 1 size: 14, Test set 2 size: 20

Best accuracy at fold 2

Best loss at fold 2
Combined Test set accuracy: 0.3333333333333333
F1-Score: 0.43572984749455335
ROC_AUC: 0.5877504848093084
Fold 3
Test set 1 size: 14, Test set 2 size: 20

Best loss at fold 3
Combined Test set accuracy: 0.3333333333333333
F1-Score: 0.43572984749455335
ROC_AUC: 0.6157886231415644
Fold 4
Test set 1 size: 14, Test set 2 size: 20

Best accuracy at fold 4

Best loss at fold 4
Combined Test set accuracy: 0.3888888888888889
F1-Score: 0.49437133343903605
ROC_AUC: 0.6553813833225597
Fold 5
Test set 1 size: 14, Test set 2 size: 20
Combined Test set accuracy: 0.3
F1-Score: 0.4072398190045249
ROC_AUC: 0.511716224951519
Fold 6
Test set 1 size: 14, Test set 2 size: 20

Best accuracy at fold 6

Best loss at fold 6
Combined Test set accuracy: 0.4305555555555556
F1-Score: 0.555